# Cohort Attrition Analysis

This notebook audits how the Synthea source population is narrowed to the final adult inpatient readmission analytic cohort. It uses local Synthea CSVs and exports aggregate-only audit outputs.

## Load SQL Cohort Pipeline

The notebook executes the same SQL views used by the project cohort construction workflow.

In [ ]:

from pathlib import Path
import sys
import os
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)

SQL_SCRIPTS = [
    "sql/01_profile_source_tables.sql",
    "sql/02_define_eligible_inpatient_encounters.sql",
    "sql/03_define_index_encounter.sql",
    "sql/04_define_postdischarge_utilization.sql",
    "sql/05_define_30_day_readmission.sql",
    "sql/06_create_final_analysis_dataset.sql",
]

con = duckdb.connect(":memory:")
for script in SQL_SCRIPTS:
    con.execute((PROJECT_ROOT / script).read_text())

def scalar(query):
    return con.execute(query).fetchone()[0]

counts = {
    "all_patients": scalar("SELECT COUNT(DISTINCT id) FROM raw_patients"),
    "patients_with_any_encounter": scalar("SELECT COUNT(DISTINCT p.id) FROM raw_patients p INNER JOIN raw_encounters e ON p.id = e.patient"),
    "patients_with_any_inpatient": scalar("SELECT COUNT(DISTINCT p.id) FROM raw_patients p INNER JOIN raw_encounters e ON p.id = e.patient WHERE LOWER(e.encounterclass) = 'inpatient'"),
    "valid_patient_birth_and_inpatient_dates": scalar("""
        SELECT COUNT(DISTINCT e.patient)
        FROM raw_encounters e
        INNER JOIN raw_patients p ON e.patient = p.id
        WHERE LOWER(e.encounterclass) = 'inpatient'
          AND e.patient IS NOT NULL
          AND TRY_CAST(p.birthdate AS DATE) IS NOT NULL
          AND TRY_CAST(e.start AS TIMESTAMP) IS NOT NULL
          AND TRY_CAST(e.stop AS TIMESTAMP) IS NOT NULL
          AND TRY_CAST(e.stop AS TIMESTAMP) >= TRY_CAST(e.start AS TIMESTAMP)
    """),
    "adult_eligible_inpatient_patients": scalar("SELECT COUNT(DISTINCT patient_id) FROM eligible_inpatient_encounters"),
    "index_admission_patients": scalar("SELECT COUNT(DISTINCT patient_id) FROM index_encounter"),
    "outcome_followup_derived_patients": scalar("SELECT COUNT(DISTINCT patient_id) FROM final_analysis_dataset WHERE readmitted_30d IS NOT NULL"),
    "final_analytic_cohort": scalar("SELECT COUNT(DISTINCT patient_id) FROM final_analysis_dataset"),
}
counts


## Build Cohort Attrition Table

Each row documents a cohort rule, the number of patients remaining, the number excluded at that step, and the exclusion reason.

In [ ]:

steps = [
    (0, "All Synthea Patients", counts["all_patients"], 0, "Starting population from patients.csv."),
    (1, "Patients With Any Encounter", counts["patients_with_any_encounter"], counts["all_patients"] - counts["patients_with_any_encounter"], "No patients excluded; every source patient had at least one encounter record."),
    (2, "Patients With Any Inpatient Encounter", counts["patients_with_any_inpatient"], counts["patients_with_any_encounter"] - counts["patients_with_any_inpatient"], "Excluded patients without an inpatient encounter because the readmission workflow requires an index hospitalization."),
    (3, "Valid Patient Linkage, Birth Date, and Inpatient Dates", counts["valid_patient_birth_and_inpatient_dates"], counts["patients_with_any_inpatient"] - counts["valid_patient_birth_and_inpatient_dates"], "No additional patients excluded; inpatient encounters linked to patients, had birth dates, valid admission dates, valid discharge dates, and discharge was not before admission."),
    (4, "Adult Patients With Eligible Inpatient Encounter", counts["adult_eligible_inpatient_patients"], counts["valid_patient_birth_and_inpatient_dates"] - counts["adult_eligible_inpatient_patients"], "Excluded patients who were under age 18 across valid inpatient encounters."),
    (5, "First Eligible Inpatient Encounter Selected as Index Admission", counts["index_admission_patients"], counts["adult_eligible_inpatient_patients"] - counts["index_admission_patients"], "No patients excluded; first eligible inpatient encounter was selected to create one index admission per patient."),
    (6, "30-Day Follow-Up and Readmission Outcome Derived", counts["outcome_followup_derived_patients"], counts["index_admission_patients"] - counts["outcome_followup_derived_patients"], "No patients excluded; 30-day post-discharge utilization and readmission fields were derived for every index patient."),
    (7, "Final Analytic Cohort", counts["final_analytic_cohort"], counts["outcome_followup_derived_patients"] - counts["final_analytic_cohort"], "Final analysis dataset rows match the dashboard KPI cohort count."),
]
attrition = pd.DataFrame(steps, columns=["Step Number", "Cohort Step", "Patients Remaining", "Patients Excluded At Step", "Exclusion Reason"])
attrition["Percent Remaining"] = (100 * attrition["Patients Remaining"] / counts["all_patients"]).round(1)
attrition = attrition[["Step Number", "Cohort Step", "Patients Remaining", "Patients Excluded At Step", "Percent Remaining", "Exclusion Reason"]]
attrition.to_csv(OUTPUT_DIR / "cohort_attrition.csv", index=False)
attrition


## Cohort Flow Dataset

This aggregate edge list can support a future Sankey or flow-chart visualization.

In [ ]:

flow = pd.DataFrame([
    {"source_step": "All Synthea Patients", "target_step": "Patients With Any Encounter", "count": counts["patients_with_any_encounter"]},
    {"source_step": "Patients With Any Encounter", "target_step": "Patients With Any Inpatient Encounter", "count": counts["patients_with_any_inpatient"]},
    {"source_step": "Patients With Any Encounter", "target_step": "Excluded: No Inpatient Encounter", "count": counts["patients_with_any_encounter"] - counts["patients_with_any_inpatient"]},
    {"source_step": "Patients With Any Inpatient Encounter", "target_step": "Valid Patient Linkage, Birth Date, and Inpatient Dates", "count": counts["valid_patient_birth_and_inpatient_dates"]},
    {"source_step": "Valid Patient Linkage, Birth Date, and Inpatient Dates", "target_step": "Adult Patients With Eligible Inpatient Encounter", "count": counts["adult_eligible_inpatient_patients"]},
    {"source_step": "Valid Patient Linkage, Birth Date, and Inpatient Dates", "target_step": "Excluded: Under Age 18 at Valid Inpatient Encounter", "count": counts["valid_patient_birth_and_inpatient_dates"] - counts["adult_eligible_inpatient_patients"]},
    {"source_step": "Adult Patients With Eligible Inpatient Encounter", "target_step": "First Eligible Inpatient Encounter Selected as Index Admission", "count": counts["index_admission_patients"]},
    {"source_step": "First Eligible Inpatient Encounter Selected as Index Admission", "target_step": "30-Day Follow-Up and Readmission Outcome Derived", "count": counts["outcome_followup_derived_patients"]},
    {"source_step": "30-Day Follow-Up and Readmission Outcome Derived", "target_step": "Final Analytic Cohort", "count": counts["final_analytic_cohort"]},
])
flow.to_csv(OUTPUT_DIR / "cohort_flow_summary.csv", index=False)
flow


## Validation Checks

These checks confirm every exclusion is accounted for and the final cohort matches the dashboard KPI cohort count.

In [ ]:

starting = counts["all_patients"]
exclusions = attrition["Patients Excluded At Step"].sum()
final = counts["final_analytic_cohort"]
dashboard_cohort = scalar("SELECT COUNT(*) FROM final_analysis_dataset")
validation = {
    "starting_patients": starting,
    "total_exclusions": int(exclusions),
    "starting_minus_exclusions": int(starting - exclusions),
    "final_analytic_cohort": final,
    "dashboard_kpi_cohort_count": dashboard_cohort,
    "all_exclusions_accounted_for": bool(starting - exclusions == final),
    "final_matches_dashboard_kpi": bool(final == dashboard_cohort),
}
validation


## Publication-Style Cohort Flow Figure

In [ ]:

import textwrap

fig, ax = plt.subplots(figsize=(12, 8), facecolor="white")
ax.set_axis_off()
ax.set_title("Cohort Attrition Audit: Adult Inpatient Readmission Cohort", loc="left", fontsize=16, fontweight="bold", color="#10233f", pad=18)
y_positions = list(reversed([0.11 + i * 0.11 for i in range(len(attrition))]))
for idx, (row, y) in enumerate(zip(attrition.itertuples(index=False), y_positions)):
    step_number = row._0
    label = row._1
    remaining = row._2
    excluded = row._3
    pct_remaining = row._4
    is_final = label == "Final Analytic Cohort"
    face = "#e8f3f1" if is_final else "#f7f9fb"
    edge = "#1f7a7a" if is_final else "#cad5df"
    box = FancyBboxPatch((0.06, y - 0.037), 0.84, 0.078, boxstyle="round,pad=0.012,rounding_size=0.012", linewidth=1.2, edgecolor=edge, facecolor=face, transform=ax.transAxes)
    ax.add_patch(box)
    wrapped_label = textwrap.fill(f"Step {step_number}: {label}", width=62)
    ax.text(0.085, y + 0.014, wrapped_label, transform=ax.transAxes, fontsize=10.2, color="#10233f", fontweight="bold", va="center")
    ax.text(0.87, y + 0.014, f"{remaining:,}", transform=ax.transAxes, fontsize=11, color="#10233f", fontweight="bold", va="center", ha="right")
    ax.text(0.085, y - 0.020, f"Remaining: {pct_remaining:.1f}% of source patients | Excluded at step: {excluded:,}", transform=ax.transAxes, fontsize=8.5, color="#52606d", va="center")
    if idx < len(y_positions) - 1:
        ax.annotate("", xy=(0.48, y_positions[idx + 1] + 0.045), xytext=(0.48, y - 0.047), xycoords=ax.transAxes, arrowprops={"arrowstyle": "->", "lw": 1.2, "color": "#52606d"})
ax.text(0.06, 0.025, "Largest cohort reduction: 878 patients without an inpatient encounter. Synthetic Synthea data only; not clinical evidence.", transform=ax.transAxes, fontsize=9, color="#52606d")
fig.savefig(FIGURE_DIR / "cohort_attrition_audit.png", dpi=180, bbox_inches="tight")
plt.show()
